****Structured Output:****
Model can be requested to provide their response in format matching a given schema
Langchain supports multiple schema types and methods for enforcing structured output 

****Pydantic****
**** ****
Pydantic models provide the richest feature set with field validation ,description ,and nested structure.

In [10]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")

model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x74ed2f7b1f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x74ed2f7b2bc0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
from pydantic import BaseModel,Field

class Movie(BaseModel):    #we Integrat BaseModel - baseclass to creating pydantic models
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="The name of the director")
    rating:float=Field(description="The movies rating out of 10")


In [ ]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure 

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x74ed2f7b1f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x74ed2f7b2bc0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'The name of the director', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'typ

In [13]:
model.invoke("Provide details about movie Sita Ramam")

AIMessage(content='<think>\nOkay, so I need to figure out the details about the movie "Sita Ramam." Let me start by recalling what I know. I think it\'s an Indian film, maybe Telugu? I remember hearing that it\'s a period romance. The title itself, "Sita Ramam," sounds like it\'s referencing the characters Sita and Rama from the Indian epic, the Ramayana. But I also think it\'s a fictional story inspired by the epic rather than a direct retelling.\n\nFirst, I should check the director. I believe the director is Dasarath, but I\'m not 100% sure. Let me verify that. Yes, Dasarath is indeed the director. Now, the cast. The lead actors are Ravi Teja and Raashi Khanna. Ravi Teja is a well-known actor in Telugu cinema, and Raashi Khanna is a popular actress in South Indian films. I think there\'s also a third actor, maybe someone like Anupam Kher? I think he plays a significant role, perhaps the British officer? Let me confirm that. Yes, Anupam Kher is in the cast, playing a British officer 

In [14]:
model_with_structure.invoke("Provide details about movie Sita Ramam")

Movie(title='Sita Ramam', year=2022, director='Hanu Raghavapudi', rating=8.5)

****Message output alongside parsed structure****

In [21]:
from pydantic import BaseModel,Field

class Movie(BaseModel):    #we Integrat BaseModel - baseclass to creating pydantic models
    """A Movie with details"""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year movie was released")
    director:str=Field(...,description="The name of the director")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide details about movie Sita Ramam")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Sita Ramam." Let me check the tools available. There\'s a Movie function that requires title, year, director, and rating. I need to find the correct information for each parameter.\n\nFirst, I know that "Sita Ramam" is a 2022 movie directed by Hanu Raghavapudi. The title is definitely "Sita Ramam." The year would be 2022. The director\'s name is Hanu Raghavapudi. As for the rating, I think it\'s around 8.5 out of 10 based on some reviews. Let me confirm that. Yes, the rating is 8.5. \n\nSo putting it all together, the parameters should be title: "Sita Ramam," year: 2022, director: "Hanu Raghavapudi," and rating: 8.5. I need to make sure all required fields are included. The required fields are title, year, director, and rating. All are present here. Now, I\'ll format this into the tool_call JSON as specified.\n', 'tool_calls': [{'id': 'te9e0pw3r', 'function': {'a

****Nested Structure****


In [26]:
from langchain_core.messages import HumanMessage
from email import message
class Actor(BaseModel):
    Name:str
    Role:str

class MovieDetails(BaseModel):
    Title:str
    Year:int
    Cast:list[Actor]
    Genres:list[str]
    Budget:float | None =Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

message=[HumanMessage("Provide Details about movie PK")]
response=model_with_structure.invoke(message)
response

MovieDetails(Title='PK', Year=2018, Cast=[Actor(Name='Aamir Khan', Role='PK'), Actor(Name='Anuskha Sharma', Role='Raanveer'), Actor(Name='Rajpal Yadav', Role="Ranveer's Friend"), Actor(Name='Sushant Singh Rajput', Role="Ranveer's Friend")], Genres=['Drama', 'Comedy'], Budget=8.5)

****TypeDict****
**** ****
When you don't need runtime validation.

In [33]:
from typing_extensions import TypedDict,Annotated


class Movie(TypedDict):    #we Integrat BaseModel - baseclass to creating pydantic models
    """A Movie with details"""
    title:Annotated[str, ...,"The title of the movie"]
    year:Annotated[int, ...,"This year movie was released"]
    director:Annotated[str, ...,"The name of the director"]
    rating:Annotated[float, ...,"The movies rating out of 10"]

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide details about movie Sita Ramam")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Sita Ramam." Let me check what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I don\'t have the specific details yet, so I need to figure them out.\n\nFirst, I should confirm the release year. I think "Sita Ramam" came out in 2022. The director is possibly S. S. Rajamouli, but I\'m not 100% sure. The rating might be around 8.5 out of 10 based on some reviews. Wait, maybe I should double-check these facts. If the director is different, that would be a problem. Also, the title is correct as "Sita Ramam." Let me make sure there\'s no confusion with another movie. Once I\'m confident about the details, I can structure the JSON with the required parameters. If I\'m unsure about any field, maybe I should note that, but the user expects the function call with the available data. Alright, let\'s proceed with the 

In [36]:
from typing_extensions import TypedDict,Annotated

from email import message
class Actor(TypedDict):
    Name:str
    Role:str

class MovieDetails(TypedDict):
    Title:str
    Year:int
    Cast:list[Actor]
    Genres:list[str]
    Budget:float | None =Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

message=[HumanMessage("Provide Details about movie PK")]
response=model_with_structure.invoke(message)
response

{'Budget': 25000000,
 'Cast': [{'Name': 'Aamir Khan', 'Role': 'PK'},
  {'Name': 'Anushka Sharma', 'Role': 'Rani'},
  {'Name': 'Sanjay Dutt', 'Role': 'Inspector Kabir'}],
 'Genres': ['Action', 'Comedy', 'Drama'],
 'Title': 'PK',
 'Year': 2018}

In [38]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

****DataClasses****
**** ****
A data class is class typically containing mainly data, although tere are not really restrictions. You create it using the @dataclass decorator

In [40]:

from langchain.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
import os
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")

from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo #Autoslects ProviderStrategy
    )
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John,john@gmail.com,(5555)123-34454"}]
})  
print(result["structured_response"])

name='John' email='john@gmail.com' phone='(5555)123-34454'


In [41]:
print(result)

{'messages': [HumanMessage(content='Extract contact info from: John,john@gmail.com,(5555)123-34454', additional_kwargs={}, response_metadata={}, id='7244d753-62b6-4bc2-8e20-c0f428c65d5e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fvrcfhxtt', 'function': {'arguments': '{"email":"john@gmail.com","name":"John","phone":"(5555)123-34454"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 286, 'total_tokens': 318, 'completion_time': 0.073164066, 'completion_tokens_details': None, 'prompt_time': 0.028411674, 'prompt_tokens_details': None, 'queue_time': 0.198170404, 'total_time': 0.10157574}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e73a5-0ce1-70c0-8295-917eb5aa7572-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@gmail.com',

In [42]:
## with DataClass

from dataclasses import dataclass
from langchain.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
import os
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")

from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo #Autoslects ProviderStrategy
    )
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John,john@gmail.com,(5555)123-34454"}]
})  
print(result["structured_response"])

ContactInfo(name='John', email='john@gmail.com', phone='(5555)123-34454')
